# Estimator Consistency — NEES, NIS, and Whether the Covariance Is Telling the Truth

Every metric in [trajectory analysis](trajectory_analysis.ipynb) and
[evaluation metrics for poses and trajectories](../vo_evaluation_metrics.ipynb)
scores a **point estimate**: ATE, RPE, sub-trajectory drift, mAA. All of them
take a list of poses and a list of ground-truth poses, and none of them ever
looks at the estimator's own uncertainty.

But a filter or a smoother does not output a trajectory. It outputs a
trajectory **and a covariance**, and the covariance is a claim: *"the truth is
within this ellipsoid."* Nothing in `evo` or `rpg_trajectory_evaluation` ever
checks that claim.

That matters, because an ATE of 0.3 m means two completely different things:

* the estimator reported $\sigma = 0.4$ m — it is accurate *and* it knows how
  accurate it is. Downstream consumers (gating, data association, loop-closure
  acceptance, sensor fusion) can trust it.
* the estimator reported $\sigma = 0.02$ m — it is **overconfident by 15×**. It
  will reject good measurements as outliers, accept false loop closures, and
  be over-weighted by anything that fuses it. It will diverge on the next
  sequence, and ATE gave no warning at all.

Consistency evaluation is the axis that catches this. It is standard in the
target-tracking literature (Bar-Shalom), routine in aerospace navigation, and
almost entirely absent from VO/SLAM benchmark tables.

**Companion docs:**

* [Trajectory analysis](trajectory_analysis.ipynb) — ATE/RPE, alignment, unobservable DoF (§1 and §10 are prerequisites for §3 below)
* [Evaluation metrics for poses and trajectories](../vo_evaluation_metrics.ipynb) — the shared residual, and the $C=-R^\top t$ convention trap that §7 mirrors
* [Error-state EKF VIO](../error_state_extended_kalman_filter_vio.ipynb) — where $P$ comes from
* [Kalman filter](../kalman_filter.ipynb) and [Extended Kalman filter](../extended_kalman_filter.ipynb) — the innovation and $S$ used in §5
* [Lie groups and Lie algebra](../lie_group_lie_algebra.ipynb) — the $\ominus$ of §2 and the adjoint of §7
* [Nonlinear uncertainty and the banana shape](../nonlinear_uncertainty_model_associated_with_robot_position_banana_shape_.ipynb) — why a Gaussian in the wrong parametrisation is already inconsistent


## 1. Accuracy and consistency are different failure modes

Four possible estimators, all with the same ATE:

| | Small error | Large error |
|---|---|---|
| **Small claimed $P$** | Good | **Overconfident** — the dangerous one |
| **Large claimed $P$** | **Conservative** — wasteful but safe | Honest about being bad |

ATE only distinguishes the columns. Consistency evaluation distinguishes the
rows.

**Why the top-right cell is the dangerous one.** Overconfidence is not a
cosmetic reporting problem — it is *self-reinforcing*:

1. $P$ is too small $\Rightarrow$ the Mahalanobis gate $\nu^\top S^{-1}\nu <
   \tau$ is too tight $\Rightarrow$ good measurements are rejected as outliers.
2. Fewer measurements accepted $\Rightarrow$ the state is corrected less
   $\Rightarrow$ the true error grows.
3. Meanwhile $P$ keeps shrinking, because the filter believes each accepted
   measurement more than it should.
4. Back to step 1, with a worse ratio.

This is the standard EKF divergence path, and it is invisible to every metric
in the two companion notebooks. §5.1 puts a number on step 1: a filter
overconfident by a factor of 4 throws away **47 % of perfectly good
measurements**.

The conservative cell (bottom-left) is not free either. If your VIO is fused
downstream with GNSS or wheel odometry, an inflated $P$ means the fusion
under-weights VIO exactly when it is right.


## 2. NEES — Normalised Estimation Error Squared

For a state estimate $\hat{x}_k$ with covariance $P_k$ and ground truth $x_k$,
define the error and its normalised square:

$$
e_k = \hat{x}_k \ominus x_k, \qquad
\boxed{\;\epsilon_k = e_k^{\top} P_k^{-1} e_k\;}
$$

This is the squared **Mahalanobis distance** of the truth from the estimate —
the same quadratic form used as a *cost* in
[factor graphs](../factor_graph.ipynb) and
[pose-graph SLAM](../pose_graph_slam.ipynb), used here as a *test* instead.

If the estimator is consistent — that is, if $e_k \sim \mathcal{N}(0, P_k)$ as
the estimator claims — then

$$
\epsilon_k \sim \chi^2_n, \qquad \mathbb{E}[\epsilon_k] = n, \qquad
\operatorname{Var}[\epsilon_k] = 2n
$$

where $n = \dim(x)$. So the headline check is: **the average NEES should equal
the state dimension.** Anything much larger means overconfident; much smaller
means conservative.

**The $\ominus$ must be the manifold error, not vector subtraction.** For a
pose $T \in SE(3)$ the error is

$$
e = \log\!\big(T_{\text{gt}}^{-1}\hat{T}\big)^{\vee} \in \mathbb{R}^6
\quad\text{(right / body-frame perturbation)}
$$

or $\log(\hat{T} T_{\text{gt}}^{-1})^{\vee}$ for the left / world-frame
convention. **$P$ must be expressed in the same convention as $e$** — this is
not a detail, it is the single most common way a NEES computation is silently
wrong (§7). Subtracting quaternions or Euler angles componentwise gives a
number that is not a NEES at all.


### 2.1 Worked example — same ATE, opposite verdict

A 2-D position state, $n = 2$. The estimate is off by

$$
e = \begin{bmatrix} 0.30 \\ -0.40 \end{bmatrix}\ \text{m},
\qquad \|e\| = 0.5\ \text{m}.
$$

The 95 % acceptance bound is $\chi^2_{2,\,0.95} = 5.991$.

**Estimator A** claims $P_A = \operatorname{diag}(0.01, 0.01)$, i.e. $\sigma = 10$ cm per axis:

$$
\epsilon_A = \frac{0.30^2}{0.01} + \frac{0.40^2}{0.01}
           = \frac{0.09}{0.01} + \frac{0.16}{0.01} = 9 + 16 = \mathbf{25}
$$

$25 \gg 5.991$ — reject. Equivalently $\sqrt{25} = 5$: the truth sits at $5\sigma$.

**Estimator B** claims $P_B = \operatorname{diag}(0.09, 0.09)$, i.e. $\sigma = 30$ cm per axis:

$$
\epsilon_B = \frac{0.09}{0.09} + \frac{0.16}{0.09} = 1 + 1.778 = \mathbf{2.78}
$$

$2.78 < 5.991$ — accept, and close to the expected value $n = 2$.

**Both estimators produced the identical trajectory.** Their ATE contribution is
the same 0.5 m. The only difference is what they claimed about themselves, and
that difference is the whole verdict. A is broken and B is fine, and no metric
in the companion notebooks can tell them apart.


### 2.2 Worked example — why a per-axis $3\sigma$ plot is not enough

The most common "consistency check" in practice is plotting each error
component against $\pm 3\sqrt{P_{ii}}$. That check is blind to the off-diagonal
terms, and the off-diagonals are where marginalisation and linearisation
errors actually live.

Take an estimator that claims $x$ and $y$ errors are 99 % correlated:

$$
P = \begin{bmatrix} 0.01 & 0.0099 \\ 0.0099 & 0.01 \end{bmatrix}
\quad (\sigma_x = \sigma_y = 0.1\ \text{m},\ \rho = 0.99)
$$

and an actual error of $e = [\,0.15,\ -0.15\,]^\top$.

**Per-axis check:** $0.15 / 0.1 = 1.5\sigma$ in $x$, $1.5\sigma$ in $y$. Both
comfortably inside the $3\sigma$ envelope. The plot looks healthy.

**NEES.** First the determinant:

$$
\det P = (0.01)(0.01) - (0.0099)^2 = 0.0001 - 0.00009801 = 1.99\times10^{-6}
$$

$$
P^{-1} = \frac{1}{1.99\times10^{-6}}
\begin{bmatrix} 0.01 & -0.0099 \\ -0.0099 & 0.01 \end{bmatrix}
$$

The quadratic form, with $e_1 = 0.15$, $e_2 = -0.15$:

$$
e^\top \!\begin{bmatrix} 0.01 & -0.0099 \\ -0.0099 & 0.01 \end{bmatrix}\! e
= 0.01(0.0225) + 2(-0.0099)(0.15)(-0.15) + 0.01(0.0225)
$$
$$
= 0.000225 + 0.0004455 + 0.000225 = 0.0008955
$$

$$
\epsilon = \frac{0.0008955}{1.99\times10^{-6}} = \mathbf{450.0}
$$

$450$ against a bound of $5.991$ — rejected by a factor of 75.

The estimator asserted that the two errors move **together** (along the $+45^\circ$
diagonal). The actual error moved **against** each other, along the direction
the covariance says is nearly impossible: the minor axis of that ellipse has
standard deviation $\sqrt{0.01(1-0.99)} = 0.01$ m, and the error has a
$0.15\sqrt{2}/\sqrt{2} = 0.15$ m component along it — $15\sigma$.

Per-axis plots cannot see this. NEES sees nothing else.


## 3. The gauge trap — the reason absolute NEES is usually meaningless

This section is the consistency-side twin of
[trajectory analysis §1](trajectory_analysis.ipynb) (why raw APE against ground
truth is huge even for a correct estimator).

A VO/SLAM state is defined only **up to an unobservable gauge transform**. The
same table applies:

| System | Unobservable DoF $g$ |
|---|---|
| Monocular VO | 7 (6-DoF pose + scale) |
| Stereo / RGB-D / LiDAR | 6 (initial pose) |
| Visual-inertial | 4 (position + yaw about gravity) |

Along those $g$ directions the estimator has **no information at all**. In a
batch/smoothing formulation the information matrix is exactly rank-deficient,
so $P = \Lambda^{-1}$ does not exist. In a filter that anchors the first pose,
$P$ exists but grows without bound along the gauge directions, and its value
there is an artefact of the anchor, not a statement about accuracy.

Consequently:

* $P^{-1}$ either does not exist or is dominated by the gauge block.
* $\epsilon = e^\top P^{-1} e$ measures **how far the estimator's arbitrary
  frame is from the dataset's arbitrary frame** — the same thing un-aligned APE
  measures, and just as uninformative.

**Two fixes, both direct analogues of episode 1's ATE/RPE split:**

| Point-estimate metric | Consistency analogue | Gauge behaviour |
|---|---|---|
| ATE / APE | Absolute NEES | Gauge-**dependent** — needs anchoring or a pseudo-inverse |
| RPE | **Relative NEES** | Gauge-**invariant** — report this one |

**Relative NEES.** For a keyframe pair $(i,j)$, form the relative pose error
and its marginal covariance,

$$
e_{ij} = \log\!\Big( \big(T_i^{-1}T_j\big)^{-1}\big(\hat{T}_i^{-1}\hat{T}_j\big) \Big)^{\vee},
\qquad
P_{ij} = J\,\begin{bmatrix} P_{ii} & P_{ij} \\ P_{ji} & P_{jj}\end{bmatrix} J^\top
$$

with $J$ the Jacobian of the relative-pose composition. Then
$\epsilon_{ij} = e_{ij}^\top P_{ij}^{-1} e_{ij} \sim \chi^2_6$.

Because $T_i^{-1}T_j$ is invariant under any global left-multiplication
$T \mapsto G\,T$, so is $e_{ij}$, and so is $\epsilon_{ij}$. **Getting the
cross-covariance $P_{ij}$ right is essential** — using
$P_{ii} + P_{jj}$ and ignoring the correlation is a very common shortcut and
it makes $P_{ij}$ far too large, i.e. it manufactures apparent consistency.

**Pseudo-inverse form.** If you must test the absolute state, restrict to the
observable subspace: with $N \in \mathbb{R}^{n \times g}$ a basis of the gauge
null space and $\Pi = I - N(N^\top N)^{-1}N^\top$ the projector onto its
complement,

$$
\epsilon^{\perp} = (\Pi e)^\top P^{\dagger} (\Pi e) \sim \chi^2_{\,n-g}
$$

using the Moore–Penrose pseudo-inverse. **Note the degrees of freedom drop from
$n$ to $n-g$** — using $\chi^2_n$ bounds on a rank-$(n-g)$ statistic is a
second, independent error, and it always errs towards declaring a broken filter
acceptable. For a single 6-DoF pose from a VIO ($g=4$): the right bound is
$\chi^2_{2,\,0.95} = 5.991$, not $\chi^2_{6,\,0.95} = 12.592$ — off by more
than $2\times$.


### 3.1 Worked example — a perfect trajectory that absolute NEES rejects

A monocular-VIO estimate that is geometrically **exact**, but whose world frame
is yawed by $\delta\psi = 2^\circ = 0.0349$ rad relative to the dataset frame —
which is the expected situation, since IMU initialisation cannot observe yaw
([trajectory analysis §10](trajectory_analysis.ipynb)).

Take a ground-truth point at $(50, 0)$ m. The estimate places it at

$$
\begin{bmatrix} 50\cos 2^\circ \\ 50 \sin 2^\circ \end{bmatrix}
= \begin{bmatrix} 49.9695 \\ 1.7452 \end{bmatrix},
\qquad
e = \begin{bmatrix} -0.0305 \\ 1.7452 \end{bmatrix}\ \text{m}
$$

With a claimed $\sigma = 0.1$ m per axis:

$$
\epsilon_{\text{abs}} = \left(\frac{0.0305}{0.1}\right)^{\!2}
                      + \left(\frac{1.7452}{0.1}\right)^{\!2}
= 0.093 + 304.57 = \mathbf{304.7}
$$

against $\chi^2_{2,\,0.95} = 5.991$. Rejected by a factor of 50.

Now the relative pose between two consecutive keyframes 1 m apart. The estimate
is geometrically exact, so

$$
\hat{T}_i^{-1}\hat{T}_j = (G T_i)^{-1}(G T_j) = T_i^{-1}T_j
\quad\Longrightarrow\quad e_{ij} = 0
\quad\Longrightarrow\quad \epsilon_{ij} = \mathbf{0}
$$

for **every** pair, regardless of $\delta\psi$.

Absolute NEES says the filter is catastrophically broken. Relative NEES says it
is perfect. Relative NEES is right — the estimator's only "error" is that it
chose a different, equally valid, gauge. Reporting absolute NEES on a system
with unobservable DoF is the consistency-side version of reporting un-aligned
APE, and it is wrong for exactly the same reason.


## 4. ANEES — and why one run tests almost nothing

A single $\epsilon_k$ is one draw from a $\chi^2_n$ with variance $2n$. For
$n = 2$ the 95 % interval of a single sample is

$$
[\chi^2_{2,\,0.025},\ \chi^2_{2,\,0.975}] = [0.051,\ 7.378]
$$

— a **145-fold** range. A single run essentially cannot reject anything short
of catastrophe.

The fix is $M$ **independent Monte-Carlo runs**, with the *average* NEES
(ANEES), normalised so that consistency means 1:

$$
\overline{\epsilon} = \frac{1}{M n}\sum_{i=1}^{M} \epsilon^{(i)},
\qquad
\text{accept if } \ \overline{\epsilon} \in
\left[\frac{\chi^2_{Mn,\,0.025}}{Mn},\ \frac{\chi^2_{Mn,\,0.975}}{Mn}\right]
$$

because $Mn\,\overline{\epsilon} \sim \chi^2_{Mn}$ under the consistency
hypothesis.

**Worked example.** $M = 50$ runs, $n = 2$, so $Mn = 100$:

$$
\chi^2_{100,\,0.025} = 74.22, \qquad \chi^2_{100,\,0.975} = 129.56
$$
$$
\Longrightarrow\quad \text{accept if } \ \overline{\epsilon} \in [\,0.742,\ 1.296\,]
$$

The interval has shrunk from 145-fold to 1.75-fold. Compare $M = 10$
($Mn = 20$): $[0.480,\ 1.709]$, still 3.6-fold. **This is why consistency is
reported from tens of runs and accuracy is reported from one** — and why
`rpg_trajectory_evaluation`'s multiple-runs-per-(algorithm, sequence) mode
exists.


### 4.1 The hard part — where do $M$ independent runs come from?

On a recorded dataset you have exactly **one** noise realisation. The IMU noise,
the photon noise, the vibration in EuRoC MH_01 were sampled once in 2015. Four
options, in decreasing order of rigour:

**(a) Simulation.** Generate ground truth, then synthesise measurements with
freshly sampled noise for each of the $M$ runs. This is the only setting where
ANEES is *exactly* the test described above, and it is how filter designers
(and every OpenVINS-style unit test) validate a new formulation. Weakness: it
validates the filter against its own noise model, so it catches linearisation
and information-double-counting bugs but not model mismatch with the real
sensor.

**(b) Seed resampling on real data.** Re-run with different RNG seeds so that
RANSAC, feature selection, and thread scheduling differ. This samples
**algorithmic** randomness, not sensor noise, so the $M$ runs are not draws from
the assumed distribution and the $\chi^2$ bounds are not exactly valid. Still
useful: if seed-to-seed spread alone already exceeds the claimed $P$, the
filter is definitively overconfident. See
[benchmark methodology §2](benchmark_methodology.ipynb) for how much this
spread actually is.

**(c) Bootstrap over sequences.** Treat each sequence's time-averaged NEES as
one sample and bootstrap the aggregate. Weak, but it is something, and it is
honest about the unit of independence.

**(d) Time-averaged NEES on a single run — and its bias.** Replacing the $M$
runs with $K$ timesteps of one run,

$$
\overline{\epsilon} = \frac{1}{Kn}\sum_{k=1}^{K}\epsilon_k,
$$

is what almost everyone actually does. It is **biased as a test**, because
consecutive $\epsilon_k$ are strongly correlated: a filter that is overconfident
at $t$ is overconfident at $t + \Delta t$. The effective sample size is
$K_{\text{eff}} \ll K$, so the $\chi^2_{Kn}$ acceptance band computed from $K$
is far too tight, and a perfectly acceptable filter gets rejected.

If you use (d) — and you probably will — **do not quote the $\chi^2_{Kn}$
band**. Quote the ratio $\overline{\epsilon}$ itself, and interpret it on a log
scale: $\overline{\epsilon} \approx 1$ good, $\approx 3$ suspicious,
$\gtrsim 10$ definitively overconfident. The magnitude is informative even when
the significance test is not.

**The fallback everyone ships: the $3\sigma$ envelope.** Plot each error
component against $\pm 3\sqrt{P_{ii}(t)}$ over time. It catches gross
overconfidence, it shows *when* the filter goes wrong (usually at a specific
manoeuvre), and it needs no Monte Carlo. It is blind to correlation errors —
see §2.2, where a 450-NEES failure sits at $1.5\sigma$ per axis. Use it as a
first look, never as the verdict.


## 5. NIS — the consistency test that needs no ground truth

NEES needs $x_k$. NIS does not, which means it runs **online, on the robot, on
data you have never seen**.

At each update, with measurement $z_k$, predicted measurement $h(\hat{x}_{k|k-1})$,
Jacobian $H_k$ and measurement noise $R_k$:

$$
\nu_k = z_k - h(\hat{x}_{k|k-1}),
\qquad
S_k = H_k P_{k|k-1} H_k^{\top} + R_k,
\qquad
\boxed{\;\text{NIS}_k = \nu_k^{\top} S_k^{-1} \nu_k\;}
$$

Under consistency $\text{NIS}_k \sim \chi^2_m$ with $m = \dim(z)$, so
$\mathbb{E}[\text{NIS}] = m$. The innovation sequence should additionally be
**white** — a significant autocorrelation in $\nu_k$ means unmodelled dynamics
(an uncompensated bias, a wrong time offset, an unmodelled lever arm), which NIS
magnitude alone will not reveal.

This is the same quadratic form the filter already computes for **outlier
gating**, so NIS costs nothing: log what you are already computing.

The two are complementary. NIS tests the *predicted measurement* distribution;
it can look perfect while the state covariance is wrong, because a filter can
be self-consistent in measurement space and still be wrong about the
unobservable parts of the state. NEES tests the state directly but needs truth.
Report NIS from field data, NEES from simulation.


### 5.1 Worked example — overconfidence throws away half the measurements

A 2-D reprojection residual, $m = 2$. The filter claims
$S = \operatorname{diag}(1, 1)\ \text{px}^2$ ($\sigma = 1$ px per axis), but the
true innovation standard deviation is 2 px per axis — the filter is
overconfident by a factor of 2 in $\sigma$, 4 in variance.

**Expected NIS.** With true innovation covariance $S_{\text{true}} = 4I$:

$$
\mathbb{E}[\text{NIS}] = \operatorname{tr}\!\big(S^{-1}S_{\text{true}}\big)
= \operatorname{tr}\!\big(I^{-1}\,4I\big) = 8
$$

against an expected $m = 2$. Ratio 4 — exactly the variance-overconfidence
factor, which is the useful diagnostic property: **the NIS ratio reads off the
overconfidence factor directly.**

**The cost at the gate.** A standard 95 % Mahalanobis gate rejects a
measurement when $\text{NIS} > \chi^2_{2,\,0.95} = 5.991$. Under the *true*
statistics, $\text{NIS} = \nu^\top I^{-1} \nu$ with $\nu \sim
\mathcal{N}(0, 4I)$, so $\text{NIS} = 4\,\chi^2_2$. The gate therefore fires
whenever

$$
4\,\chi^2_2 > 5.991 \iff \chi^2_2 > 1.4978
$$

and for two degrees of freedom the survival function is
$P(\chi^2_2 > x) = e^{-x/2}$:

$$
P(\text{reject}) = e^{-1.4978/2} = e^{-0.7489} = \mathbf{0.473}
$$

**47 % of perfectly ordinary measurements are discarded as outliers.** A single
concrete case: $\nu = [\,2,\ -2\,]^\top$ px is a routine $1\sigma$ observation
under the true noise, and it scores $\text{NIS} = 4 + 4 = 8 > 5.991$ —
rejected.

The filter now updates on barely half its measurements, so the true error
grows, while $P$ continues to shrink on the ones it does accept. That is step 1
of the divergence loop in §1, quantified.


## 6. Diagnosis — what actually causes each direction

### Overconfident: $\overline{\epsilon} \gg 1$

| Cause | Mechanism | Fix |
|---|---|---|
| **Observability inconsistency** | The EKF linearises the *same* state at *different* estimates across timesteps. The linearised system then has fewer unobservable directions than the true one, so the filter **gains information about yaw and global position that it never measured**. $P$ shrinks along directions where no information exists. This is the classic EKF-SLAM inconsistency result (Julier & Uhlmann 2001; Bailey 2006; Huang, Mourikis & Roumeliotis 2010) and it is *structural* — it happens with a perfectly correct implementation. | **FEJ** (First-Estimates Jacobians) or OC-EKF; both are options in [OpenVINS](../open_vins.md). Or use a smoother, where relinearisation is global. |
| **Linearisation error** | Jacobians evaluated far from the truth; large rotations between updates; the [banana-shaped](../nonlinear_uncertainty_model_associated_with_robot_position_banana_shape_.ipynb) true distribution approximated by an ellipse. | Smaller propagation steps, iterated EKF, on-manifold parametrisation. |
| **Information double-counting** | The same landmark's measurements enter through both the VO front-end and a loop-closure factor; or a loop-closure constraint is computed *from* the odometry it then constrains. Each reuse shrinks $P$ as if it were new evidence. | Track measurement provenance; never close a loop with a constraint derived from the trajectory being corrected. |
| **Naive sparsification** | Marginalising a keyframe produces a **dense** prior. Dropping the off-diagonal blocks to keep the graph sparse discards correlations, which is optimistic. | Consistent sparsification (Chow–Liu tree approximations), or keep the dense prior. |
| **Under-modelled process noise** | $Q$ too small: gyro/accel bias random walk taken from an ideal datasheet rather than from an Allan-variance fit of *your* unit at *your* temperature. | Measure it — see [IMU](../imu.ipynb) and [Kalibr](../kalibr.md). |
| **Unmodelled time offset** | A camera–IMU $t_d$ of a few ms appears as a *bias* in the innovations, not as noise. NIS whiteness catches this; NIS magnitude alone may not. | Estimate $t_d$ online (OpenVINS, VINS-Fusion both do). |

### Conservative: $\overline{\epsilon} \ll 1$

| Cause | Mechanism |
|---|---|
| **Inflated $R$ "for robustness"** | The most common hack in the field. It suppresses divergence by making the filter ignore its measurements, and it works — at the cost of accuracy, and of lying to anything downstream that fuses this output. |
| **Robust-cost double-counting** | A Huber/Cauchy kernel already down-weights outliers; inflating $R$ on top of it down-weights twice. |
| **Covariance inflation / fading memory** | Deliberate $P \leftarrow \alpha P$, $\alpha > 1$. Legitimate, but it must be declared — the reported $P$ is then not a posterior covariance. |

A conservative filter is safe on its own but is a **liar in a fusion stack**: a
downstream EKF weights it by $P^{-1}$ and will under-use a good sensor.


## 7. Where $P$ comes from — and the convention trap

Neither `evo` nor `rpg_trajectory_evaluation` accepts a covariance. Both take
poses and nothing else, so none of this section's tests can be run through
them. You have to get $P$ out of the estimator yourself.

| Source | How | Notes |
|---|---|---|
| ROS 1/2 | `geometry_msgs/PoseWithCovarianceStamped` | $6\times6$ row-major, order $(x,y,z,\text{rot}_x,\text{rot}_y,\text{rot}_z)$. The rotation block's frame is **not specified by the message** — read the publisher. |
| OpenVINS | `/ov_msckf/poseimu` (`PoseWithCovarianceStamped`), plus `save_total_state` for the full state and covariance | Covariance is in the **IMU/body** frame, right perturbation |
| GTSAM | `Marginals(graph, values).marginalCovariance(key)` | `Pose3` uses the **body-frame right** perturbation $T\exp(\xi^\wedge)$ |
| Ceres | `ceres::Covariance` | **Fails on a rank-deficient problem** — you must fix the gauge first (a prior on pose 0, or `SetParameterBlockConstant`), which is §3 in code form |
| g2o | `computeMarginals()` | Requires the same gauge fix; see [g2o](../g2o.md) |

### The trap: $P$ lives in a parametrisation, and so must $e$

A $6\times6$ covariance is meaningless without three declarations: **left or
right** perturbation, **which frame** the translation block is in, and
**translation-first or rotation-first** ordering. Getting any one wrong
produces a NEES that is confidently, silently wrong. Converting a body-frame
covariance to the world frame requires the adjoint:

$$
P_{\text{world}} = \operatorname{Ad}_{T}\, P_{\text{body}} \operatorname{Ad}_{T}^{\top},
\qquad
\operatorname{Ad}_{T} = \begin{bmatrix} R & [t]_\times R \\ 0 & R \end{bmatrix}
$$

**Worked example.** A ground vehicle: uncertainty is large along-track and small
cross-track, so in the **body** frame

$$
P_{\text{body}} = \operatorname{diag}(0.25,\ 0.0025)
\qquad (\sigma_{\text{along}} = 0.5\ \text{m},\ \sigma_{\text{cross}} = 0.05\ \text{m}).
$$

The vehicle is currently driving along the world $y$ axis, i.e. yawed $90^\circ$
from world. Then

$$
P_{\text{world}} = R\,P_{\text{body}}R^{\top} = \operatorname{diag}(0.0025,\ 0.25)
$$

— the axes swap. Now suppose the actual error, in world coordinates, is
$e = [\,0,\ 0.3\,]^\top$ m, i.e. 30 cm **along** the direction of travel.

$$
\text{correct:}\quad \epsilon = \frac{0}{0.0025} + \frac{0.09}{0.25} = \mathbf{0.36}
\quad\Longrightarrow\quad \text{consistent } (< 5.991)
$$

$$
\text{using } P_{\text{body}} \text{ as if it were world:}\quad
\epsilon = \frac{0}{0.25} + \frac{0.09}{0.0025} = \mathbf{36}
\quad\Longrightarrow\quad \text{rejected}
$$

A factor of **100** in the statistic, and opposite verdicts, from a single
missing adjoint — and nothing in the output looks wrong. This is the same class
of error as the $C = -R^{\top}t$ trap in
[evaluation metrics §3](../vo_evaluation_metrics.ipynb): a convention mismatch
that produces plausible numbers rather than a crash.


## 8. Reporting recipe

Alongside the ATE/RPE/drift table from
[trajectory analysis §8](trajectory_analysis.ipynb), report:

1. **Which NEES** — relative (gauge-invariant, preferred) or absolute with an
   explicit statement of the anchoring or the projector $\Pi$ and pseudo-inverse.
2. **The degrees of freedom used**, and the gauge dimension $g$ subtracted.
   $n - g$, not $n$.
3. **$\overline{\epsilon}$ normalised so consistency $= 1$**, with $M$, and the
   $\chi^2$ acceptance interval **only if the $M$ samples are genuinely
   independent** (§4.1(a)). If they are timesteps of one run, say so and drop
   the interval.
4. **The perturbation convention and frame** of $P$ — left/right,
   body/world, and the block ordering.
5. **NIS from field runs**, with the innovation autocorrelation, and the
   measured outlier-rejection rate. A rejection rate far above the gate's
   nominal $\alpha$ is the cheapest overconfidence detector there is.
6. **The $\pm3\sigma$ error-vs-time plot** per axis, as the qualitative
   companion — it shows *when*, which the scalar does not.
7. Whether any covariance inflation, fading memory, or robust kernel is active.

**Minimum viable version**, if you do nothing else: log NIS at every update,
plot its running mean against $m$, and report the gate rejection rate. It is
free, it needs no ground truth, and it catches the failure mode that ATE
structurally cannot see.


## 9. References

* Y. Bar-Shalom, X.-R. Li, T. Kirubarajan, *Estimation with Applications to
  Tracking and Navigation*, Wiley 2001 — Ch. 5: NEES, NIS, ANEES, and the
  $\chi^2$ acceptance regions. The origin of everything in §2, §4, §5.
* S. Julier, J. Uhlmann, *A counter example to the theory of simultaneous
  localization and map building*, ICRA 2001 — the first demonstration that
  EKF-SLAM is inconsistent by construction.
* T. Bailey, J. Nieto, J. Guivant, M. Stevens, E. Nebot, *Consistency of the
  EKF-SLAM algorithm*, IROS 2006.
* G. Huang, A. Mourikis, S. Roumeliotis, *Observability-based rules for
  designing consistent EKF SLAM estimators*, IJRR 29(5), 2010 — the FEJ result.
* J. Hesch, D. Kottas, S. Bowman, S. Roumeliotis, *Camera-IMU-based
  localization: observability analysis and consistency improvement*, IJRR 2014
  — the VIO (4-DoF) version.
* P. Geneva, K. Eckenhoff, W. Lee, Y. Yang, G. Huang, *OpenVINS: a research
  platform for visual-inertial estimation*, ICRA 2020 — FEJ as a shipped option.
* Z. Zhang, D. Scaramuzza, *A tutorial on quantitative trajectory evaluation
  for visual(-inertial) odometry*, IROS 2018 — the unobservable-DoF taxonomy
  reused in §3.

## 10. See also

* [Trajectory analysis](trajectory_analysis.ipynb) — ATE/RPE, alignment, and the unobservable-DoF table that §3 builds on
* [Evaluation metrics for poses and trajectories](../vo_evaluation_metrics.ipynb) — the shared residual and the convention trap §7 mirrors
* [Benchmark methodology](benchmark_methodology.ipynb) — run-to-run variance, which §4.1(b) needs
* [Runtime evaluation](runtime_evaluation.ipynb) — the third axis after accuracy and consistency
* [Error-state EKF VIO](../error_state_extended_kalman_filter_vio.ipynb) · [Kalman filter](../kalman_filter.ipynb) · [Extended Kalman filter](../extended_kalman_filter.ipynb)
* [Factor graph](../factor_graph.ipynb) · [Pose-graph SLAM](../pose_graph_slam.ipynb) — the same Mahalanobis form, used as a cost
* [Diagnostic procedure for broken VIO](../../vio_benchmark/docs/VIO_DIAGNOSTIC_GUIDE.md)
